### Today we are going to perform the simple classification of the amazon reviews' sentiment.

### Please, download the dataset amazon_baby.csv.


**Note:** Place `amazon_baby.csv` in the same folder as this notebook before running.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

baby_df = pd.read_csv('amazon_baby.csv')
baby_df.head()

,name,review,rating
0,Planetwise Flannel Wipes,"These flannel wipes are OK, but in my opinion ...",3
1,Planetwise Wipe Pouch,it came early and was not disappointed. i love...,5
2,Annas Dream Full Quilt with 2 Shams,Very soft and comfortable and warmer than it l...,5
3,Stop Pacifier Sucking without tears with Thumb...,This is a product well worth the purchase. I ...,5
4,Stop Pacifier Sucking without tears with Thumb...,All of my kids have cried non-stop when I trie...,5


## Exercise 1 (data preparation)
a) Remove punctuation from reviews using the given function.   
b) Replace all missing (nan) reviews with empty "" string.  
c) Drop all the entries with rating = 3, as they have neutral sentiment.   
d) Set all positive ($\geq$4) ratings to 1 and negative($\leq$2) to -1.

In [2]:
# a) Remove punctuation from every review (missing values are left as they are and handled in b).
baby_df["review"] = baby_df["review"].apply(
    lambda text: remove_punctuation(text) if isinstance(text, str) else text
)

# short test: no punctuation left in a sample review
not any(ch in string.punctuation for ch in baby_df["review"][4])

True

In [3]:
# b) Replace missing reviews with empty strings.
baby_df["review"] = baby_df["review"].fillna("")

# short test: the number of missing reviews should be 0
baby_df["review"].isna().sum()

np.int64(0)

In [4]:
# c) Remove neutral (3-star) reviews.
baby_df = baby_df[baby_df["rating"] != 3].copy()

# short test (should be 0):
sum(baby_df["rating"] == 3)


0

In [5]:
# d) Convert ratings to sentiment labels: positive = 1, negative = -1.
baby_df["rating"] = np.where(baby_df["rating"] >= 4, 1, -1)

# short test (should be 0):
sum(baby_df["rating"]**2 != 1)


0

## CountVectorizer
In order to analyze strings, we need to assign them numerical values. We will use one of the simplest string representations, which transforms strings into the $n$ dimensional vectors. The number of dimensions will be the size of our dictionary, and then the values of the vector will represent the number of appearances of the given word in the sentence.

In [6]:
reviews_train_example = ["We like apples",
                         "We hate oranges",
                         "I adore bananas",
                         "We like apples and like oranges",
                         "They dislike bananas"]

vectorizer = CountVectorizer()
X_train_example = vectorizer.fit_transform(reviews_train_example)

In [7]:
print(vectorizer.get_feature_names_out())
print(X_train_example.todense())


['adore' 'and' 'apples' 'bananas' 'dislike' 'hate' 'like' 'oranges' 'they'
 'we']
[[0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 1 0 1 0 1]
 [1 0 0 1 0 0 0 0 0 0]
 [0 1 1 0 0 0 2 1 0 1]
 [0 0 0 1 1 0 0 0 1 0]]


In [8]:
reviews_test_example = ["They like bananas",
                   "We hate oranges bananas and apples",
                   "We love bananas"] #New word!

X_test_example = vectorizer.transform(reviews_test_example)

print(X_test_example.todense())

[[0 0 0 1 0 0 1 0 1 0]
 [0 1 1 1 0 1 0 1 0 1]
 [0 0 0 1 0 0 0 0 0 1]]


We should acknowledge a few facts. Firstly, CountVectorizer does not take order into account. Secondly, it ignores one-letter words (this can be changed during initialization). Finally, for test values, CountVectorizer ignores words which are not in its dictionary.

## Exercise 2 
a) Split dataset into training and test sets.     
b) Transform reviews into vectors using CountVectorizer. 

In [9]:
# a) Split reviews and labels into training and test sets.
from sklearn.model_selection import train_test_split

X_text = baby_df["review"]
y = baby_df["rating"]
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train_text))
print("Test rows:", len(X_test_text))


Training rows: 133401
Test rows: 33351


In [10]:
# b) Fit CountVectorizer only on training reviews and transform both sets.
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print("Training matrix shape:", X_train.shape)
print("Test matrix shape:", X_test.shape)


Training matrix shape: (133401, 121546)
Test matrix shape: (33351, 121546)


## Exercise 3 
a) Train LogisticRegression model on training data (reviews processed with CountVectorizer, ratings as they were).   
b) Print 10 most positive and 10 most negative words.

In [11]:
# a) Train logistic regression on the training data.
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Model trained.")


Model trained.


In [12]:
# b) Print 10 most positive and 10 most negative words.
words = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]
positive_indices = np.argsort(coefficients)[-10:][::-1]
negative_indices = np.argsort(coefficients)[:10]
print("10 most positive words:")
for i in positive_indices:
    print(words[i], coefficients[i])
print("\n10 most negative words:")
for i in negative_indices:
    print(words[i], coefficients[i])


10 most positive words:
lifesaver 2.181680140241267
penny 2.1529567734141986
thankful 2.1528389983454836
ply 2.054453352131923
posture 2.0143314473229337
rich 1.9934605132124215
con 1.9559215603708042
pleasantly 1.9185853083958384
worry 1.8962258608031066
minor 1.8695020034493253

10 most negative words:
dissapointed -2.8107480003650593
worst -2.7248676794083417
disappointing -2.649688527965246
unusable -2.605057871908057
worthless -2.5406939024500694
poorly -2.4485148972425637
useless -2.420428339151592
concept -2.396648434062004
poor -2.3230372828589863
ineffective -2.3095702684979424


## Exercise 4 
a) Predict the sentiment of test data reviews.   
b) Predict the sentiment of test data reviews in terms of probability.   
c) Find five most positive and most negative reviews.   
d) Calculate the accuracy of predictions.

In [13]:
# a) Predict the sentiment labels for test reviews.
y_pred = model.predict(X_test)
print(y_pred[:20])


[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [14]:
# b) Predict probabilities for negative (-1) and positive (1) sentiment.
y_proba = model.predict_proba(X_test)
print("Class order:", model.classes_)
print(y_proba[:10])


Class order: [-1  1]
[[1.47221796e-02 9.85277820e-01]
 [8.98688113e-02 9.10131189e-01]
 [1.17165382e-01 8.82834618e-01]
 [6.81769240e-02 9.31823076e-01]
 [6.77746371e-02 9.32225363e-01]
 [2.81690310e-03 9.97183097e-01]
 [9.50353445e-04 9.99049647e-01]
 [8.51946228e-04 9.99148054e-01]
 [2.10936564e-03 9.97890634e-01]
 [8.91381939e-03 9.91086181e-01]]


In [15]:
# c) Find five most positive and five most negative test reviews.
# Probability of the positive class (1) is used to rank the reviews.
positive_class_index = list(model.classes_).index(1)
positive_prob = y_proba[:, positive_class_index]
most_positive_indices = np.argsort(positive_prob)[-5:][::-1]
most_negative_indices = np.argsort(positive_prob)[:5]
print("FIVE MOST POSITIVE REVIEWS:")
for idx in most_positive_indices:
    print(f"Positive probability: {positive_prob[idx]:.4f} | Review: {X_test_text.iloc[idx]}\n")
print("FIVE MOST NEGATIVE REVIEWS:")
for idx in most_negative_indices:
    print(f"Positive probability: {positive_prob[idx]:.4f} | Review: {X_test_text.iloc[idx]}\n")


FIVE MOST POSITIVE REVIEWS:
Positive probability: 1.0000 | Review: I went back to work full time just six weeks after my son was born so pumping has been a regular part of my life for months now Ive tried all different sorts of storage bags and bottle organizing kits but this system really caught my attention Heres how it worked for usKoziiIll start with this since its the most wellknown item in the kit My husband and our nanny both LOVE this bottle warmer Its extremely easy to use and takes all the guesswork out of warming food for my son You just fill it with water to the clearlymarked level indicator plug it in and set the timer Thats it It will happily warm milk formula baby food or whatever fits into itTwist PouchesI love these bags so much The screwtop lid makes feeding a snap and even if you dont plan to use the included bottles pouring milk OUT of these bags is much easier than other bags weve tried I freeze in 5 oz portions then put ten bags into a big freezer bag for easy tra

In [16]:
# d) Calculate the accuracy of predictions.
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")


Accuracy: 0.9307 (93.07%)


## Exercise 5
In this exercise we will limit the dictionary of CountVectorizer to the set of significant words, defined below.


a) Redo exercises 2-4 using limited dictionary.   
b) Check the impact of all the words from the dictionary.   
c) Compare accuracy of predictions and the time of evaluation.

In [17]:
significant_words = ['love','great','easy','old','little','perfect','loves','well','able','car','broke','less','even','waste','disappointed','work','product','money','would','return']

In [18]:
# a) Repeat exercises 2-4 using only the significant words.
limited_vectorizer = CountVectorizer(vocabulary=significant_words)
X_train_limited = limited_vectorizer.fit_transform(X_train_text)
X_test_limited = limited_vectorizer.transform(X_test_text)
limited_model = LogisticRegression(max_iter=1000)
limited_model.fit(X_train_limited, y_train)
limited_words = limited_vectorizer.get_feature_names_out()
limited_coef = limited_model.coef_[0]
print("Limited training matrix:", X_train_limited.shape)
print("Limited test matrix:", X_test_limited.shape)
print("10 most positive words:")
for i in np.argsort(limited_coef)[-10:][::-1]:
    print(limited_words[i], limited_coef[i])
print("\n10 most negative words:")
for i in np.argsort(limited_coef)[:10]:
    print(limited_words[i], limited_coef[i])
# Predictions and probabilities for limited vocabulary
limited_pred = limited_model.predict(X_test_limited)
limited_proba = limited_model.predict_proba(X_test_limited)
limited_positive_index = list(limited_model.classes_).index(1)
limited_positive_prob = limited_proba[:, limited_positive_index]
print("Limited-vocabulary accuracy:", accuracy_score(y_test, limited_pred))

# Five most positive / negative test reviews for the limited vocabulary
print("\nFIVE MOST POSITIVE REVIEWS (limited vocabulary):")
for idx in np.argsort(limited_positive_prob)[-5:][::-1]:
    print(f"Positive probability: {limited_positive_prob[idx]:.4f} | Review: {X_test_text.iloc[idx]}\n")
print("FIVE MOST NEGATIVE REVIEWS (limited vocabulary):")
for idx in np.argsort(limited_positive_prob)[:5]:
    print(f"Positive probability: {limited_positive_prob[idx]:.4f} | Review: {X_test_text.iloc[idx]}\n")


Limited training matrix: (133401, 20)
Limited test matrix: (33351, 20)
10 most positive words:
loves 1.7472868666292265
perfect 1.4745915357354422
love 1.3967027963078222
easy 1.1722470710271806
great 0.934330463249113
little 0.5195562170676359
well 0.4863393465211001
able 0.20821249860033247
old 0.08382291569708568
car 0.05902060918400964

10 most negative words:
disappointed -2.4386030045638742
return -2.141940486683007
waste -1.9825148637442382
broke -1.6696901910418416
money -0.9258621274971152
work -0.6045059870167111
even -0.5336584138772388
would -0.34910210282847404
product -0.32410034793931947
less -0.19313184030351888
Limited-vocabulary accuracy: 0.8667206380618272

FIVE MOST POSITIVE REVIEWS (limited vocabulary):
Positive probability: 1.0000 | Review: My daughter 8 months old loves this toy and has for a few months I love it too  Its not only a stacking toy but several toys in one each providing development benefits The removable elephant head serves as a funnel into which y

In [19]:
# b) Check the impact (coefficient) of every word in the limited dictionary.
word_impact = pd.DataFrame({
    "word": limited_words,
    "coefficient": limited_coef
}).sort_values("coefficient", ascending=False)
word_impact


,word,coefficient
6,loves,1.747287
5,perfect,1.474592
0,love,1.396703
2,easy,1.172247
1,great,0.934330
4,little,0.519556
7,well,0.486339
8,able,0.208212
3,old,0.083823
9,car,0.059021


In [20]:
# c) Compare accuracy and evaluation time for the full and limited dictionaries.
# %time measures one execution, while %timeit repeats it to get a more reliable estimate.

# Full dictionary: prediction time (single run) and repeated timing.
%time full_pred_timed = model.predict(X_test)
full_timeit = %timeit -o model.predict(X_test)
full_accuracy = accuracy_score(y_test, full_pred_timed)

# Limited dictionary: prediction time (single run) and repeated timing.
%time limited_pred_timed = limited_model.predict(X_test_limited)
limited_timeit = %timeit -o limited_model.predict(X_test_limited)
limited_accuracy = accuracy_score(y_test, limited_pred_timed)

comparison = pd.DataFrame({
    "Dictionary": ["Full", "Limited (significant words)"],
    "Number of features": [X_train.shape[1], X_train_limited.shape[1]],
    "Accuracy": [full_accuracy, limited_accuracy],
    "Prediction time (s, mean)": [full_timeit.average, limited_timeit.average]
})
comparison

CPU times: total: 15.6 ms
Wall time: 8.01 ms
5.91 ms ± 68.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
CPU times: total: 15.6 ms
Wall time: 628 μs
549 μs ± 10.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


,Dictionary,Number of features,Accuracy,"Prediction time (s, mean)"
0,Full,121546,0.930707,0.005914
1,Limited (significant words),20,0.866721,0.000549
